In [ ]:
!pip install tensorflow scikit-learn pandas numpy -q

In [ ]:
from google.colab import files
import pandas as pd
from sklearn.model_selection import train_test_split

uploaded = files.upload()

df = pd.read_csv('raid_balanced.csv')
print(f"Shape: {df.shape}")
print(df['label'].value_counts())

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 20000
MAX_LEN = 200

# Fit tokenizer
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['text'].tolist())

# Convert to sequences
def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts.tolist())
    return pad_sequences(seqs, maxlen=MAX_LEN,
                        padding='pre',
                        truncating='pre')

X_train = encode(train_df['text'])
X_val   = encode(val_df['text'])
X_test  = encode(test_df['text'])

y_train = np.array(train_df['label'])
y_val   = np.array(val_df['label'])
y_test  = np.array(test_df['label'])

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Sample X_train[0] first 10: {X_train[0][:10]}")
print(f"Zeros at start: {(X_train[0][:20]==0).sum()}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, Bidirectional,
                                      LSTM, Dense, Dropout,
                                      BatchNormalization)

tf.random.set_seed(42)

model = Sequential([
    Embedding(MAX_WORDS, 64),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Build model explicitly
model.build(input_shape=(None, MAX_LEN))
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_raw = model.predict(X_test, verbose=0)
y_pred = (y_pred_raw > 0.5).astype(int)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred,
      target_names=['Human', 'AI']))

# Verify prediction distribution
print(f"\nPredictions > 0.5 (AI): {(y_pred_raw > 0.5).sum()}")
print(f"Predictions < 0.5 (Human): {(y_pred_raw < 0.5).sum()}")
print("This is Bilstm")

In [ ]:
from google.colab import drive
import pickle

drive.mount('/content/drive')

model.save('/content/drive/MyDrive/bilstm_model.keras')
with open('/content/drive/MyDrive/bilstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Saved successfully!")

In [ ]:
def predict_text(text):
    padded = encode(pd.Series([text]))
    pred = model.predict(padded, verbose=0)[0][0]

    if pred > 0.5:
        label = "🤖 AI Generated"
        confidence = pred
    else:
        label = "👤 Human Written"
        confidence = 1 - pred

    print(f"Prediction : {label}")
    print(f"Confidence : {confidence*100:.2f}%")
    print(f"Raw score  : {pred:.4f}")
    print("-" * 40)

# Use actual samples from dataset
print("👤 HUMAN TEXTS:")
for i in [0, 4, 6]:
    text = test_df[test_df['label']==0]['text'].iloc[i]
    print(f"Text: {text[:80]}...")
    predict_text(text)

print("\n🤖 AI TEXTS:")
for i in [0, 1, 2]:
    text = test_df[test_df['label']==1]['text'].iloc[i]
    print(f"Text: {text[:80]}...")
    predict_text(text)

In [ ]:
# Run this to get actual test samples
print("👤 HUMAN - copy and paste these:")
human_samples = test_df[test_df['label']==0]['text'].iloc[0:3]
for i, text in enumerate(human_samples):
    print(f"\nSample {i+1}:")
    print(text)
    print("---")

print("\n\n🤖 AI - copy and paste these:")
ai_samples = test_df[test_df['label']==1]['text'].iloc[0:3]
for i, text in enumerate(ai_samples):
    print(f"\nSample {i+1}:")
    print(text)
    print("---")

In [ ]:
while True:
    text = input("\nEnter text (or 'quit'): ")
    if text.lower() == 'quit':
        print("Exiting...")
        break
    predict_text(text)